In [5]:
import torch
from torch import nn
from torch.nn import functional as F



#resnet结构设计
class Residual(nn.Module):  #@save
    def __init__(self, input_channels, num_channels,
                 use_1x1conv=False, strides=1):
        super().__init__()
        self.conv1 = nn.Conv2d(input_channels, num_channels,
                               kernel_size=3, padding=1, stride=strides)
        self.conv2 = nn.Conv2d(num_channels, num_channels,
                               kernel_size=3, padding=1)
        if use_1x1conv:
            self.conv3 = nn.Conv2d(input_channels, num_channels,
                                   kernel_size=1, stride=strides)
        else:
            self.conv3 = None
        #固定维度
        self.bn1 = nn.BatchNorm2d(num_channels)
        self.bn2 = nn.BatchNorm2d(num_channels)

    def forward(self, X):
        Y = F.relu(self.bn1(self.conv1(X)))
        Y = self.bn2(self.conv2(Y))
        if self.conv3:
            X = self.conv3(X)
        Y += X
        return F.relu(Y)

b1 = nn.Sequential(
    nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3),
    nn.BatchNorm2d(64), nn.ReLU(),
    nn.MaxPool2d(kernel_size=3, stride=2, padding=1))
def resnet_block(input_channels, num_channels, num_residuals,
                 first_block=False):
    blk = []
    for i in range(num_residuals):
        if i == 0 and not first_block:
            blk.append(Residual(input_channels, num_channels,
                                use_1x1conv=True, strides=2))
        else:
            blk.append(Residual(num_channels, num_channels))
    return blk
b2 = nn.Sequential(*resnet_block(64, 64, 2, first_block=True))
b3 = nn.Sequential(*resnet_block(64, 128, 2))
b4 = nn.Sequential(*resnet_block(128, 256, 2))
b5 = nn.Sequential(*resnet_block(256, 512, 2))

net = nn.Sequential(b1, b2, b3, b4, b5,
                    nn.AdaptiveAvgPool2d((1,1)),
                    nn.Flatten(), 
                    nn.Linear(512, 10))



In [ ]:
# 数据载入
from torchvision import datasets
from torchvision import transforms
from torch.utils.data import DataLoader
from torchvision.datasets import Imagenette

image_mean = [0.485,0.456,0.406]
image_std = [0.229,0.224,0.225]

# 训练集的数据处理
train_transform = transforms.Compose([
    # 随机裁剪出224×224图像，同时包含随机缩放
    transforms.RandomResizedCrop(224),

    # 以0.5的概率进行水平翻转
    transforms.RandomHorizontalFlip(p=0.5),

    # 将PIL图像转换为Tensor
    transforms.ToTensor(),

    # 对三个通道分别标准化
    transforms.Normalize(
        mean=image_mean,
        std=image_std
    )
])
val_transform = transforms.Compose([
    # 先保持比例，将短边缩放到256
    transforms.Resize(256),

    # 从图像中心裁剪224×224
    transforms.CenterCrop(224),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=image_mean,
        std=image_std
    )
])

#创建训练集和交叉验证集
train_dataset = Imagenette(root='./data', 
        split='train', 
        download=True,
        size = 'full',
        transform = train_transform)
val_dataset = Imagenette(root='./data', 
        split='val', 
        download=True,
        size = 'full',
        transform = val_transform)
#加载训练数据集和测试数据集
batch_size = 128
#保持打乱，
train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    drop_last=False,
    prefetch_factor= 1
)
val_loader = DataLoader(
    dataset=val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    drop_last=False,
    prefetch_factor= 1
)

In [ ]:
from pathlib import Path
from torch.utils.tensorboard import SummaryWriter

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

net = net.to(device)
loss_fn = nn.CrossEntropyLoss()


#优化器，沿用sgd，momentum 0.9
optimizer = torch.optim.SGD(
    net.parameters(),
    lr=0.1,
    momentum=0.9,
    weight_decay=1e-4
)

#tensorboard画笔
writer = SummaryWriter(
    log_dir="./logs/resnet18_imagenette"
)

#在gpu下使用混合精度，提升运算效率，gpu有
#  矩阵加速运算，cpu没有，反而会慢
use_amp = device.type == "cuda"

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=use_amp
)

#创建保存模型的文件夹
checkpoint_dir = Path("./checkpoints")
checkpoint_dir.mkdir(parents=True, exist_ok=True)


#训练参数
epochs = 30
global_step = 0
best_val_accuracy = 0.0



# =========================================================
# 9. 开始训练
# =========================================================
for epoch in range(epochs):
    print(f"\n========== Epoch {epoch + 1}/{epochs} ==========")

    # -----------------------------------------------------
    # 训练阶段
    # -----------------------------------------------------
    net.train()

    train_loss_sum = 0.0
    train_correct = 0
    train_total = 0

    for batch_idx, (images, labels) in enumerate(train_loader):
        images = images.to(
            device,
            non_blocking=True
        )

        labels = labels.to(
            device,
            non_blocking=True
        )

        # 清空上一批数据的梯度
        optimizer.zero_grad(set_to_none=True)

        # 混合精度前向传播
        with torch.autocast(
            device_type=device.type,
            dtype=torch.float16,
            enabled=use_amp
        ):
            outputs = net(images)
            loss = loss_fn(outputs, labels)

        # 反向传播
        scaler.scale(loss).backward()

        # 更新模型参数
        scaler.step(optimizer)

        # 更新GradScaler的缩放比例
        scaler.update()

        # 当前batch的图片数量
        current_batch_size = images.size(0)

        # 累积损失
        ##损失计算是当前batchsize的平均损失
        train_loss_sum += loss.item() * current_batch_size




        # outputs形状为 [batch_size, 10]
        predictions = outputs.argmax(dim=1)
        ###这里输出为[batch_size,1]




        #当轮训练正确率
        train_correct += (
            predictions == labels
        ).sum().item()


        ##当轮训练图片数量
        train_total += current_batch_size


        ##每一个batchsize加1
        global_step += 1

        # 每20个batch打印和记录一次
        if global_step % 20 == 0:
            ##当前batch的正确率，item()只取数字
            batch_accuracy = (
                predictions == labels
            ).float().mean().item()
            writer.add_scalar(
                "train/batch_loss",
                loss.item(),
                global_step
            )


            ##写入每个batchsize的正确率
            writer.add_scalar(
                "train/batch_accuracy",
                batch_accuracy,
                global_step
            )

    # 当前epoch的平均训练损失
    train_loss = train_loss_sum / train_total

    # 当前epoch的训练准确率
    train_accuracy = train_correct / train_total
    # -----------------------------------------------------
    # 验证阶段
    # -----------------------------------------------------
    ##表示模型进入验证集阶段
    net.eval()

    val_loss_sum = 0.0
    val_correct = 0
    val_total = 0

    # 验证时不需要计算梯度
    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(
                device,
                non_blocking=True
            )

            labels = labels.to(
                device,
                non_blocking=True
            )

            with torch.autocast(
                device_type=device.type,
                dtype=torch.float16,
                enabled=use_amp
            ):
                outputs = net(images)
                loss = loss_fn(outputs, labels)

            current_batch_size = images.size(0)


            ##计算当轮训练损失
            val_loss_sum += (
                loss.item() * current_batch_size
            )

            ##训练轮预测
            predictions = outputs.argmax(dim=1)


            ##当轮训练集正确数量
            val_correct += (
                predictions == labels
            ).sum().item()


            ##当轮训练集总共图片
            val_total += current_batch_size

    val_loss = val_loss_sum / val_total
    val_accuracy = val_correct / val_total

    ##写入每轮误差
    writer.add_scalars(
        "epoch loss",
        {'train':train_loss,
         'val':val_loss},
        epoch + 1
    )
    ##写入每轮正确率
    writer.add_scalars(
        "epoch accuracy",
        {'train':train_accuracy,
         'val':val_accuracy},
        epoch + 1
    )


    # -----------------------------------------------------
    # 保存最佳模型
    # -----------------------------------------------------
    if val_accuracy > best_val_accuracy:
        best_val_accuracy = val_accuracy

        torch.save(
            {
                "epoch": epoch + 1,
                "model_state_dict": net.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "val_accuracy": val_accuracy
            },
            checkpoint_dir / "best_resnet18_imagenette.pth"
        )

        print(
            f"保存最佳模型，验证准确率："
            f"{best_val_accuracy:.4f}"
        )
writer.close()


========== Epoch 1/30 ==========


d:\miniconda\envs\6661kai\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


KeyboardInterrupt: 

In [ ]:
##继续训练几个循环

checkpoint_path = checkpoint_dir / "best_resnet18_imagenette.pth"

checkpoint = torch.load(
    checkpoint_path,#加载的文件路径
    map_location=device,#放到哪个设备上
    weights_only=False#加载除了模型参数外其他默认值
                      #如优化器，损失值等
)


#创建画笔
writer = SummaryWriter(
    log_dir=f"./logs/resnet_process"
)


#加载模型（net）参数
net.load_state_dict(
    checkpoint["model_state_dict"]
)
#加载优化器参数
optimizer.load_state_dict(
    checkpoint["optimizer_state_dict"]
)
#恢复之前的epoch
start_epoch = checkpoint["epoch"]
# 找到
# 找到最佳验证准确率
best_val_accuracy = checkpoint["val_accuracy"]

#设定结束周期
additional_epochs = 20
end_epoch = start_epoch + additional_epochs

#总batichsize数量
global_step = start_epoch * len(train_loader)
# =========================================================
# 9. 开始训练
# =========================================================
for epoch in range(start_epoch,end_epoch):
    print(f"\n========== Epoch {epoch + 1}/{epochs} ==========")

    # -----------------------------------------------------
    # 训练阶段
    # -----------------------------------------------------
    net.train()

    train_loss_sum = 0.0
    train_correct = 0
    train_total = 0

    for batch_idx, (images, labels) in enumerate(train_loader):
        images = images.to(
            device,
            non_blocking=True
        )

        labels = labels.to(
            device,
            ##non_blocking表示数据传输和程序其他操作可以重叠进行
            non_blocking=True
        )

        # 清空上一批数据的梯度
        optimizer.zero_grad(set_to_none=True)

        # 混合精度前向传播
        with torch.autocast(
            device_type=device.type,
            dtype=torch.float16,
            enabled=use_amp
        ):
            outputs = net(images)
            loss = loss_fn(outputs, labels)

        # 反向传播
        scaler.scale(loss).backward()

        # 更新模型参数
        scaler.step(optimizer)

        # 更新GradScaler的缩放比例
        scaler.update()

        # 当前batch的图片数量
        current_batch_size = images.size(0)

        # 累积损失
        ##损失计算是当前batchsize的平均损失
        train_loss_sum += loss.item() * current_batch_size




        # outputs形状为 [batch_size, 10]
        predictions = outputs.argmax(dim=1)
        ###这里输出为[batch_size,1]




        #当轮训练正确率
        train_correct += (
            predictions == labels
        ).sum().item()


        ##当轮训练图片数量
        train_total += current_batch_size


        ##每一个batchsize加1
        global_step += 1

        # 每20个batch打印和记录一次
        if global_step % 20 == 0:
            ##当前batch的正确率，item()只取数字
            batch_accuracy = (
                predictions == labels
            ).float().mean().item()
            writer.add_scalar(
                "train/batch_loss",
                loss.item(),
                global_step
            )


            ##写入每个batchsize的正确率
            writer.add_scalar(
                "train/batch_accuracy",
                batch_accuracy,
                global_step
            )

    # 当前epoch的平均训练损失
    train_loss = train_loss_sum / train_total

    # 当前epoch的训练准确率
    train_accuracy = train_correct / train_total
    # -----------------------------------------------------
    # 验证阶段
    # -----------------------------------------------------
    ##表示模型进入验证集阶段
    net.eval()

    val_loss_sum = 0.0
    val_correct = 0
    val_total = 0

    # 验证时不需要计算梯度
    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(
                device,
                non_blocking=True
            )

            labels = labels.to(
                device,
                non_blocking=True
            )

            with torch.autocast(
                device_type=device.type,
                dtype=torch.float16,
                enabled=use_amp
            ):
                outputs = net(images)
                loss = loss_fn(outputs, labels)

            current_batch_size = images.size(0)


            ##计算当轮训练损失
            val_loss_sum += (
                loss.item() * current_batch_size
            )

            ##训练轮预测
            predictions = outputs.argmax(dim=1)


            ##当轮训练集正确数量
            val_correct += (
                predictions == labels
            ).sum().item()


            ##当轮训练集总共图片
            val_total += current_batch_size

    val_loss = val_loss_sum / val_total
    val_accuracy = val_correct / val_total

#写入epoch损失
    writer.add_scalars(
        "epoch loss",
        {'train':train_loss,
         'val':val_loss},
        epoch + 1
    )
    ##写入每轮正确率
    writer.add_scalars(
        "epoch accuracy",
        {'train':train_accuracy,
         'val':val_accuracy},
        epoch + 1
    )
    # -----------------------------------------------------
    # 保存最佳模型
    # -----------------------------------------------------
    if val_accuracy > best_val_accuracy:
        best_val_accuracy = val_accuracy

        torch.save(
            {
                "epoch": epoch + 1,
                "model_state_dict": net.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "val_accuracy": val_accuracy
            },
            checkpoint_dir / "best_resnet18_imagenette.pth"
        )

        print(
            f"保存最佳模型，验证准确率："
            f"{best_val_accuracy:.4f}"
        )
writer.close()


#### 加入tencrop和学习率变化

In [ ]:
image_mean = [0.485,0.456,0.406]
image_std = [0.229,0.224,0.225]

# 训练集的数据处理
normalize = transforms.Normalize(
    mean=image_mean,
    std=image_std
)
train_transform = transforms.Compose([
    # 随机裁剪出224×224图像，同时包含随机缩放
    transforms.RandomResizedCrop(224),

    # 以0.5的概率进行水平翻转
    transforms.RandomHorizontalFlip(p=0.5),

    # 将PIL图像转换为Tensor
    transforms.ToTensor(),

    # 对三个通道分别标准化
    normalize()
])
val_transform = transforms.Compose([
    transforms.Resize(256),
    ##加入tencrop
    transforms.TenCrop(224),
    transforms.Lambda(
        lambda crops: torch.stack([
            normalize(transforms.ToTensor()(crop))
            for crop in crops
        ])
    )
])

#创建训练集和交叉验证集
train_dataset = Imagenette(root='./data', 
        split='train', 
        download=True,
        size = 'full',
        transform = train_transform)
val_dataset = Imagenette(root='./data', 
        split='val', 
        download=True,
        size = 'full',
        transform = val_transform)
#加载训练数据集和测试数据集
batch_size = 128
#保持打乱，
train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    drop_last=False,
    prefetch_factor= 1
)
val_loader = DataLoader(
    dataset=val_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    drop_last=False,
    prefetch_factor= 1
)


In [ ]:
net = nn.Sequential(b1, b2, b3, b4, b5,
                    nn.AdaptiveAvgPool2d((1,1)),
                    nn.Flatten(), 
                    nn.Linear(512, 10))
net = net.to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(
    net.parameters(),
    lr=0.01,
    momentum=0.9,
    weight_decay=1e-4
)
writer = SummaryWriter(
    log_dir="./logs/resnet_renew"
)

##加入学习率衰减
scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer,
    step_size=20,
    gamma=0.1
)


epoch = 50
for epoch in range(epoch):
    # 训练阶段
    net.train()
    train_loss_sum = 0.0
    train_correct = 0
    train_total = 0
    for images, labels in train_loader:
        images = images.to(
            device,
            non_blocking=True
        )
        labels = labels.to(
            device,
            non_blocking=True
        )

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(
            device_type=device.type,
            dtype=torch.float16,
            enabled=use_amp
        ):
            outputs = net(images)
            loss = loss_fn(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        current_batch_size = images.size(0)
        predictions = outputs.argmax(dim=1)
        train_correct += (
            predictions == labels
        ).sum().item()
        train_total += current_batch_size
        global_step += 1
    train_accuracy = train_correct / train_total


    # 验证阶段
    net.eval()
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(
                device,
                non_blocking=True
            )

            labels = labels.to(
                device,
                non_blocking=True
            )
            batch_size_now, num_crops, channels, height, width = (
                images.shape
            )
            images = images.reshape(
                batch_size_now * num_crops,
                channels,
                height,
                width
            )
            with torch.autocast(
                device_type=device.type,
                dtype=torch.float16,
                enabled=use_amp
            ):
                outputs = net(images)
                outputs = outputs.reshape(
                    batch_size_now,
                    num_crops,
                    -1
                )
                outputs = outputs.mean(dim=1)
                loss = loss_fn(outputs, labels)

            current_batch_size = images.size(0)


            predictions = outputs.argmax(dim=1)
            val_correct += (
                predictions == labels
            ).sum().item()
            val_total += current_batch_size
    val_accuracy = val_correct / val_total


    ##写入每轮正确率
    writer.add_scalars(
        "epoch accuracy_renew",
        {'train':train_accuracy,
         'val':val_accuracy},
        epoch + 1
    )

    # 保存模型之后，再更新下一轮使用的学习率
    scheduler.step()

writer.close()
